In [ ]:

# import pandas as pd
# import numpy as np
# import os

# # Load pressure analysis dataset
# df = pd.read_csv("processed/pressure_analysis.csv")

# # Check required columns
# required = [
#     "bowler_id", "bowler_name", "match_id",
#     "runs", "extras_type", "is_wicket",
#     "dismissal_kind", "pressure"
# ]

# for col in required:
#     if col not in df.columns:
#         raise ValueError(f"Missing column: {col}")

# # Clean data
# df["runs"] = pd.to_numeric(
#     df["runs"], errors="coerce"
# ).fillna(0)

# df["extras_type"] = (
#     df["extras_type"].fillna("").astype(str).str.lower()
# )

# df["is_wicket"] = (
#     df["is_wicket"].astype(str).str.lower()
#     .isin(["true", "1", "yes"])
# )

# df["pressure"] = (
#     df["pressure"].astype(str).str.lower()
#     .isin(["true", "1", "yes"])
# )

# # Legal balls: exclude wides and no-balls
# df["legal_ball"] = (
#     ~df["extras_type"].str.contains(
#         "wide|no.?ball", regex=True, na=False
#     )
# ).astype(int)

# # Runs charged to bowler
# df["bowler_runs"] = df["runs"]

# df.loc[
#     df["extras_type"].str.contains(
#         "bye|leg.?bye|penalty", regex=True, na=False
#     ),
#     "bowler_runs"
# ] = df["bowler_runs"] - 0

# # Wickets credited to bowler
# not_bowler_wickets = [
#     "run out",
#     "retired hurt",
#     "retired out",
#     "obstructing the field"
# ]

# df["bowler_wicket"] = (
#     df["is_wicket"]
#     & ~df["dismissal_kind"].str.lower().isin(
#         not_bowler_wickets
#     )
# ).astype(int)

# # Calculate overall performance
# def calculate_stats(data, prefix):
#     return data.groupby(
#         ["bowler_id", "bowler_name"]
#     ).agg(
#         runs=("bowler_runs", "sum"),
#         balls=("legal_ball", "sum"),
#         matches=("match_id", "nunique"),
#         wickets=("bowler_wicket", "sum")
#     ).reset_index().rename(columns={
#         "runs": prefix + "_runs",
#         "balls": prefix + "_balls",
#         "matches": prefix + "_matches",
#         "wickets": prefix + "_wickets"
#     })

# overall = calculate_stats(df, "overall")

# # Calculate pressure performance
# pressure = calculate_stats(
#     df[df["pressure"]], "pressure"
# )

# result = pressure.merge(
#     overall,
#     on=["bowler_id", "bowler_name"],
#     how="left"
# )

# # Calculate economy and wicket rates
# result["pressure_economy"] = (
#     result["pressure_runs"] * 6 /
#     result["pressure_balls"].replace(0, np.nan)
# )

# result["overall_economy"] = (
#     result["overall_runs"] * 6 /
#     result["overall_balls"].replace(0, np.nan)
# )

# result["pressure_wicket_rate"] = (
#     result["pressure_wickets"] * 6 /
#     result["pressure_balls"].replace(0, np.nan)
# )

# result["overall_wicket_rate"] = (
#     result["overall_wickets"] * 6 /
#     result["overall_balls"].replace(0, np.nan)
# )

# # Compare pressure performance with baseline
# result["economy_improvement"] = (
#     result["overall_economy"] -
#     result["pressure_economy"]
# )

# result["wicket_rate_uplift"] = (
#     result["pressure_wicket_rate"] -
#     result["overall_wicket_rate"]
# )

# # Minimum eligibility
# result = result[
#     (result["pressure_balls"] >= 60) &
#     (result["pressure_matches"] >= 5)
# ].copy()

# # Standard score
# def zscore(series):
#     sd = series.std(ddof=0)

#     if pd.isna(sd) or sd == 0:
#         return pd.Series(0.0, index=series.index)

#     return (series - series.mean()) / sd

# # Weighted bowler score
# result["economy_z"] = -zscore(
#     result["pressure_economy"]
# )

# result["wicket_z"] = zscore(
#     result["pressure_wicket_rate"]
# )

# result["economy_improvement_z"] = zscore(
#     result["economy_improvement"]
# )

# result["wicket_uplift_z"] = zscore(
#     result["wicket_rate_uplift"]
# )

# raw_score = (
#     0.40 * result["economy_z"] +
#     0.25 * result["wicket_z"] +
#     0.20 * result["economy_improvement_z"] +
#     0.15 * result["wicket_uplift_z"]
# )

# # Confidence adjustment
# confidence = (
#     result["pressure_balls"] /
#     (result["pressure_balls"] + 60)
# ) * (
#     result["pressure_matches"] /
#     (result["pressure_matches"] + 3)
# )

# result["rating"] = raw_score * confidence

# # Rank bowlers
# result = result.sort_values(
#     ["rating", "pressure_balls"],
#     ascending=[False, False]
# ).reset_index(drop=True)

# result.insert(0, "rank", range(1, len(result) + 1))

# # Save Top 10
# os.makedirs("output", exist_ok=True)

# result.head(10).to_csv(
#     "output/top_10_bowlers.csv",
#     index=False
# )

# print("\nTOP 10 BOWLERS")
# print(
#     result[
#         ["rank", "bowler_name", "pressure_wickets",
#          "pressure_balls", "pressure_economy",
#          "pressure_wicket_rate", "rating"]
#     ].head(10).to_string(index=False)
# )

# print("\nBowler ranking completed!")
# print("Saved: output/top_10_bowlers.csv")
